In [0]:

bronze_path   = 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/bronze/'
silver_path   = 'abfss://bikesales@stgdatalakebr01.dfs.core.windows.net/silver/'

In [0]:
display(dbutils.fs.ls(silver_path))

In [0]:

##spark.table("bikesales.silver.orders").printSchema()



In [0]:
from pyspark.sql.functions import (
    col, when, current_timestamp, trim, concat, lit, upper, lower, 
    datediff, round, year, month, to_date, current_date
)
from pyspark.sql.types import IntegerType



In [0]:
# ============================================
# 1. Tabela orders
# ============================================
df_orders = spark.table("bikesales.bronze.orders")

df_orders_silver = (
    df_orders
    .withColumn("order_status_text",
        when(col("order_status") == 1, "Pending")
        .when(col("order_status") == 2, "Processing")
        .when(col("order_status") == 3, "Rejected")
        .when(col("order_status") == 4, "Completed")
        .otherwise("Unknown")
    )
    .withColumn("shipped_date", 
        when(col("shipped_date") == "NULL", None)
        .otherwise(to_date(col("shipped_date")))
    )
    .withColumn("tempo_entrega_dias", 
        when(col("shipped_date").isNotNull() & col("order_date").isNotNull(),
             datediff(col("shipped_date"), col("order_date"))
        ).otherwise(None)
    )
    .drop("order_status")
    .drop("data_carga_bronze")
    .withColumn("data_carga_silver", current_date())
)

df_orders_silver.write.mode("overwrite").saveAsTable("bikesales.silver.orders")

In [0]:
# 2. customers (clientes)
df_customers = spark.table("bikesales.bronze.customers")
df_customers_silver = (
    df_customers
    .withColumn("full_name", trim(concat(col("first_name"), lit(" "), col("last_name"))))
    .withColumn("state", upper(col("state")))
    .withColumn("email", lower(col("email")))
    .withColumn("phone", when(col("phone").isNull(), "N/A").otherwise(col("phone")))
    .drop("data_carga_bronze")
    .withColumn("data_carga_silver", current_date())
)
df_customers_silver.write.mode("overwrite").saveAsTable("bikesales.silver.customers")

In [0]:
# 3. brands (marcas)
df_brands = spark.table("bikesales.bronze.brands")
df_brands_silver = (
    df_brands
    .drop("data_carga_bronze")
    .withColumn("data_carga_silver", current_date())
)
df_brands_silver.write.mode("overwrite").saveAsTable("bikesales.silver.brands")

In [0]:
# 4. products (produtos)
df_products = spark.table("bikesales.bronze.products")
df_products_silver = (
    df_products
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("list_price", round(col("list_price"), 2))
    .withColumn("price_range", 
        when(col("list_price") < 500, "Budget")
        .when(col("list_price") < 1000, "Economy")
        .when(col("list_price") < 2000, "Standard")
        .when(col("list_price") < 5000, "Premium")
        .otherwise("Elite")
    )
    .drop("data_carga_bronze")
    .withColumn("data_carga_silver", current_date())
)
df_products_silver.write.mode("overwrite").saveAsTable("bikesales.silver.products")

In [0]:
# ============================================
# 5. Tabela order_items (limpeza de dados)
# ============================================
df_order_items = spark.table("bikesales.bronze.order_items")

df_order_items_silver = (
    df_order_items
    .withColumn("discount", when(col("discount").isNull(), 0).otherwise(col("discount")))
    .withColumn("quantity", when(col("quantity") <= 0, 1).otherwise(col("quantity")))
    .withColumn("valor_total", col("quantity") * col("list_price") * (1 - col("discount")))
    .drop("data_carga_bronze")
    .withColumn("data_carga_silver", current_date())
)

df_order_items_silver.write.mode("overwrite").saveAsTable("bikesales.silver.order_items")

In [0]:
# 6. staffs (funcionários) 
df_staffs = spark.table("bikesales.bronze.staffs")

df_staffs_silver = (
    df_staffs
    .withColumn("full_name", trim(concat(col("first_name"), lit(" "), col("last_name"))))
    .withColumn("email", lower(col("email")))
    .withColumn("active", col("active").cast(IntegerType()))
    .withColumn("active_status", when(col("active") == 1, "Active").otherwise("Inactive"))
  
    .withColumn("manager_id", 
        when(col("manager_id") == "NULL", None)
        .otherwise(col("manager_id").cast(IntegerType()))
    )
    .drop("data_carga_bronze")
    .drop("first_name", "last_name")
    .withColumn("data_carga_silver", current_date())
)

df_staffs_silver.write.mode("overwrite").saveAsTable("bikesales.silver.staffs")

In [0]:
# 7. stores (lojas)
df_stores = spark.table("bikesales.bronze.stores")
df_stores_silver = (
    df_stores
    .withColumn("store_name", trim(col("store_name")))
    .withColumn("state", upper(col("state")))
    .withColumn("email", lower(col("email")))
    .withColumn("phone", when(col("phone").isNull(), "N/A").otherwise(col("phone")))
    .drop("data_carga_bronze")
    .withColumn("data_carga_silver", current_date())
)
df_stores_silver.write.mode("overwrite").saveAsTable("bikesales.silver.stores")

In [0]:
# 8. stocks (estoque)
df_stocks = spark.table("bikesales.bronze.stocks")
df_stocks_silver = (
    df_stocks
    .withColumn("quantity", when(col("quantity") < 0, 0).otherwise(col("quantity")))
    .withColumn("stock_status",
        when(col("quantity") == 0, "Out of Stock")
        .when(col("quantity") < 10, "Low Stock")
        .when(col("quantity") < 50, "Normal Stock")
        .otherwise("High Stock")
    )
    .drop("data_carga_bronze")
    .withColumn("data_carga_silver", current_date())
)
df_stocks_silver.write.mode("overwrite").saveAsTable("bikesales.silver.stocks")

In [0]:
# 9. categories (categorias de produtos)
df_categories = spark.table("bikesales.bronze.categories")
df_categories_silver = (
    df_categories
    .withColumn("category_name", trim(col("category_name")))
    .withColumn("category_type",
        when(col("category_name").contains("Bicycles"), "Bicycle")
        .when(col("category_name").contains("Bikes"), "Bike")
        .otherwise("Other")
    )
    .drop("data_carga_bronze")
    .withColumn("data_carga_silver", current_date())
)
df_categories_silver.write.mode("overwrite").saveAsTable("bikesales.silver.categories")